[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/07_transformer/07_transformer_solutions.ipynb)

# 07. Transformer — 연습 문제 해설

[본문 노트북](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/07_transformer/07_transformer.ipynb)의 연습 문제 4개에 대한
정답 코드와 해설입니다.

**먼저 직접 풀어본 뒤에 보세요.** 이 노트북은 정답지라기보다 **내 코드와 비교해볼 자료**입니다.
결과 숫자가 여기와 조금 달라도 정상입니다. 봐야 할 것은 **차이의 방향과 크기**입니다.

## 이 노트북을 읽는 법

- 아래 준비 셀에 **본문의 모델·데이터 코드가 그대로 다시 들어 있습니다.**
  본문을 먼저 실행할 필요 없이 이 노트북만 열어도 됩니다.
- 네 문제는 서로 독립적이라 원하는 것만 실행해도 됩니다.
- 학습이 여러 번 반복되므로 **전부 실행하면 CPU에서 6~8분** 걸립니다.
  가장 오래 걸리는 곳은 연습 문제 1번의 다중 시드 셀입니다.

---

## 준비 — 본문의 코드 다시 싣기

본문 3·5·7·8·9절에서 만든 것을 한 셀에 모았습니다. 설명은 본문을 보세요.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q torch matplotlib numpy koreanize-matplotlib

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 데이터 (본문 9절) ────────────────────────────────────────────
DAYS = ["월요일", "화요일", "수요일", "목요일", "금요일"]
NAMES = [("민수", "는", "가"), ("지훈", "은", "이"), ("서연", "은", "이"),
         ("하나", "는", "가"), ("유진", "은", "이")]
FOODS = [("김밥", "을"), ("라면", "을"), ("냉면", "을"), ("비빔밥", "을"),
         ("피자", "를"), ("만두", "를"), ("국수", "를"), ("김치", "를")]


def make_corpus(n_sentences):
    lines = []
    for _ in range(n_sentences):
        day = random.choice(DAYS)
        name, josa1, josa2 = random.choice(NAMES)
        food, josa3 = random.choice(FOODS)
        lines.append(f"{day}에 {name}{josa1} {food}{josa3} 먹었습니다. "
                     f"{name}{josa2} 먹은 것은 {food}입니다.")
    return "\n".join(lines) + "\n"


random.seed(0)
text = make_corpus(900)

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

BLOCK_SIZE = 64
BATCH_SIZE = 32

data = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]


def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)


lines = [l for l in text.split("\n") if l]
avg_len = sum(len(l) + 1 for l in lines) / len(lines)
LOSS_FLOOR = (math.log(5) + math.log(5) + math.log(8)) / avg_len

print("글자 종류 수:", vocab_size, " loss 하한:", round(LOSS_FLOOR, 4))

In [ ]:
# ── 모델 (본문 5·7·8절) ─────────────────────────────────────────
# 연습 문제 2·3번에서 마스크와 위치 임베딩을 꺼야 하므로,
# 본문 코드에 use_mask / use_pos 스위치만 더했다. 켜두면 본문과 완전히 같다.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size, use_mask=True):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.use_mask = use_mask
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.last_attn = None

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)

        def split_heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if self.use_mask:
            scores = scores.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        self.last_attn = weights.detach()
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, block_size, use_mask=True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_head, block_size, use_mask)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.ReLU(), nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=64, n_head=4, n_layer=2,
                 use_mask=True, use_pos=True):
        super().__init__()
        self.block_size = block_size
        self.use_pos = use_pos
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.Sequential(
            *[TransformerBlock(d_model, n_head, block_size, use_mask) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.use_pos:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.ln_f(self.blocks(x))
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


@torch.no_grad()
def generate(model, prompt, n_new_chars, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(n_new_chars):
        logits, _ = model(idx[:, -BLOCK_SIZE:])
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
    return decode(idx[0].tolist())

In [ ]:
def train(label, steps=1000, seed=0, **kwargs):
    """모델을 하나 만들어 학습시키고 val loss를 돌려준다. 조건만 바꿔가며 부를 수 있다."""
    torch.manual_seed(seed)
    model = TinyGPT(vocab_size=vocab_size, block_size=BLOCK_SIZE, **kwargs).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

    model.train()
    for _ in range(steps):
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        val = sum(model(*get_batch("val"))[1].item() for _ in range(30)) / 30
    print(f"{label:26s} val loss {val:.4f}")
    return model, val


# 비교 기준이 될 모델. 이 아래 문제들은 전부 이 숫자와 비교한다.
base_model, base_val = train("기준 (2층, 4헤드)")
print(f"{'':26s} (이론적 하한 {LOSS_FLOOR:.4f})")

---

## 연습 문제 1 — 층과 헤드를 줄여보기

> `n_layer=1`과 `n_head=1`을 각각 학습시켜 val loss를 비교하세요.
> 둘 중 어느 쪽이 더 나빠지나요? 그리고 **그 차이를 믿어도 될까요?**

먼저 문제가 시킨 대로 한 번씩만 돌려봅니다.

In [ ]:
_, val_1layer = train("1층 (n_layer=1)", n_layer=1)
_, val_1head = train("1헤드 (n_head=1)", n_head=1)

print()
print("기준 대비 나빠진 정도")
print(f"  1층  : {val_1layer - base_val:+.4f}")
print(f"  1헤드: {val_1head - base_val:+.4f}")

### 잠깐 — 이 차이를 믿어도 될까요?

제가 이 노트북을 만들면서 실제로 겪은 일을 그대로 적겠습니다.

**처음 한 번 돌렸을 때는 1층이 0.2129로 기준(0.1911)보다 확실히 나빴습니다.**
"역시 유도 헤드는 두 층이 필요하구나" 하고 해설을 다 써놨습니다.

**그런데 다시 돌렸더니 1층이 0.1875로 기준(0.1922)보다 오히려 좋았습니다.**
정반대 결론이 나온 것입니다.

무엇이 문제였을까요? **차이가 0.002~0.02 수준인데, 실행할 때마다 흔들리는 폭도 그 정도**라는
것을 확인하지 않은 것입니다. 조건 때문에 생긴 차이인지 그냥 운인지 구분할 수 없었습니다.

**그래서 시드를 바꿔가며 여러 번 재야 합니다.** 아래 셀은 조건마다 시드 3개로 돌립니다
(CPU에서 4~5분 걸립니다). 스텝 수는 위의 기준 모델과 똑같이 맞춥니다 —
**학습량이 다르면 조건을 비교하는 것이 아니게 되기 때문입니다.**

In [ ]:
import statistics as st

SEEDS = [0, 1, 2]
conditions = {
    "기준 (2층 4헤드)": {},
    "1층 (n_layer=1)": {"n_layer": 1},
    "1헤드 (n_head=1)": {"n_head": 1},
}

results = {}
for name, kwargs in conditions.items():
    # 스텝 수는 위의 기준 모델과 똑같이 맞춘다. 학습량이 다르면 조건 비교가 아니게 된다.
    results[name] = [train(f"  {name} seed={s}", seed=s, **kwargs)[1] for s in SEEDS]

print()
print(f"{'조건':20s} {'평균':>8s} {'최소':>8s} {'최대':>8s} {'폭':>8s}")
for name, vals in results.items():
    print(f"{name:20s} {st.mean(vals):8.4f} {min(vals):8.4f} {max(vals):8.4f} {max(vals) - min(vals):8.4f}")
print(f"\n(이론적 하한 {LOSS_FLOOR:.4f})")

### 해설

제가 **시드 5개**로 잰 결과입니다(위 셀은 시간을 아끼려고 3개만 돌립니다.
시드 0~2만 쓰면 기준 0.1887, 1층 0.1893, 1헤드 0.2020이 나옵니다).

| 조건 | 평균 | 표준편차 | 최소~최대 |
|---|---:|---:|---|
| 기준 (2층 4헤드) | 0.1885 | 0.0029 | 0.1844~0.1922 |
| 1층 | 0.1907 | 0.0051 | 0.1853~0.1971 |
| **1헤드** | **0.1977** | **0.0132** | 0.1869~**0.2194** |
| (참고) 4층 | 0.1870 | 0.0011 | 0.1855~0.1885 |
| (이론적 하한) | 0.1424 | | |

**결론이 처음 예상과 반대입니다.**

**① 1층과 기준의 차이는 잡음입니다.** 평균 차이가 +0.0022인데 기준 자체가 실행마다
±0.003씩 흔들립니다. **차이보다 흔들림이 큽니다.** 1층 모델의 생성 문장도 멀쩡합니다 —
앞뒤 음식이 잘 맞습니다.

왜일까요? **이 데이터가 너무 규칙적이기 때문입니다.** 모든 문장이 같은 틀이라
"약 19글자 앞을 보라"는 **위치 규칙만으로도** 되풀이를 해낼 수 있습니다.
내용을 보고 되짚는 진짜 유도 헤드는 **문장 구조가 제각각일 때** 비로소 필요해집니다.

**② 진짜 차이는 헤드 쪽에 있고, 평균보다 흔들림에서 드러납니다.**
1헤드는 평균도 나쁘지만(+0.0092) **표준편차가 기준의 4배가 넘습니다**(0.0132 vs 0.0029).
최악의 시드에서는 0.2194까지 튑니다.

헤드가 여러 개라는 것은 **같은 일을 여러 번 시도하는 것**과 비슷합니다. 한 헤드가 쓸모없는
패턴에 자리를 잡아도 다른 헤드가 건집니다. 헤드가 하나면 **그 한 번에 전부를 겁니다.**
평균 성능만큼이나 **안정성**이 걸린 문제인 것입니다.

> **여기서 배울 점 두 가지**
>
> 1. **한 번 돌린 결과로 조건을 비교하면 안 됩니다.** 조건 간 차이가 실행 간 흔들림보다
>    작으면 아무 말도 할 수 없습니다. `tabular-ml-practice` 03번에서
>    [교차 검증](https://github.com/karzit/temp/blob/master/glossary.md#cross-validation)을 배운 것과 **같은 이야기**입니다 —
>    거기서는 "한 번의 분할을 믿지 말 것"이었고, 여기서는 "한 번의 학습을 믿지 말 것"입니다.
> 2. **평균만 보지 말고 흔들림도 보세요.** 1헤드의 진짜 문제는 "조금 나쁘다"가 아니라
>    **"운이 나쁘면 크게 나쁘다"**입니다. 평균만 봤다면 놓쳤을 것입니다.

**참고**: 4층으로 늘리면 평균이 조금 좋아지고(0.1870) 흔들림이 확 줄지만(0.0011),
하한(0.1424)에는 여전히 못 미칩니다. 남은 간격은 **모델 크기가 아니라 학습량**의 문제입니다.
스텝을 늘려보면 확인할 수 있습니다.

---

## 연습 문제 2 — 마스크를 빼보기

> `masked_fill` 줄을 빼고 학습시켜 loss를 보세요.
> 9절에서 계산한 **하한 0.142와 비교**하면 무슨 일이 벌어졌는지 알 수 있습니다.

In [ ]:
no_mask_model, val_nomask = train("마스크 없음", use_mask=False)

print()
print(f"이론적 하한       : {LOSS_FLOOR:.4f}")
print(f"마스크 없는 모델  : {val_nomask:.4f}")
print(f"하한보다 낮은가?  : {val_nomask < LOSS_FLOOR}")

### 해설

제가 돌렸을 때 val loss가 **0.0031**이었습니다. 이론적 하한 0.1424의 **50분의 1**입니다.

**이건 "아주 잘 배웠다"가 아니라 "부정행위를 했다"는 뜻입니다.**

9절에서 우리는 이렇게 계산했습니다. 요일·이름·음식은 무작위로 골랐으니
**아무리 똑똑한 모델도 찍을 수밖에 없고**, 그래서 loss가 0.1424 아래로는 못 내려간다고요.
그 계산은 틀리지 않았습니다. 틀린 것은 모델의 조건입니다.

**마스크를 빼면 모델이 정답을 볼 수 있습니다.** `월요일에 지훈은 김`에서 다음 글자를
맞혀야 하는데, 어텐션이 문장 전체를 보므로 **바로 그 자리의 `밥`을 그냥 읽어옵니다.**
맞히는 게 아니라 베끼는 것입니다.

**이 문제가 무서운 이유는 loss가 좋아 보인다는 것입니다.** 학습 곡선도 예쁘고,
train과 val이 둘 다 낮아서 [과적합](https://github.com/karzit/temp/blob/master/glossary.md#overfitting)처럼 보이지도 않습니다.
**9절에서 하한을 미리 계산해두지 않았다면 알아채기 어렵습니다.**

> 이것이 tabular-ml-practice 01번에서 다룬 [데이터 누출](https://github.com/karzit/temp/blob/master/glossary.md#data-leakage)과 **정확히 같은 종류의 사고**입니다.
> 거기서는 `alive` 컬럼이 정답을 흘렸고, 여기서는 마스크를 안 걸어서 흘렸습니다.
> **"성능이 너무 좋으면 의심하라"**는 규칙이 여기서도 그대로 통합니다.

진짜로 정답을 베끼고 있는지, 생성을 시켜서 확인해봅시다.

In [ ]:
print(generate(no_mask_model, "월요일에 ", 200))

**결과 읽는 법** — **문장이 엉망입니다.** loss가 0.003인데도 그렇습니다.

이유는 단순합니다. 생성할 때는 **아직 안 쓴 미래 글자가 없습니다.**
학습 내내 미래를 읽어서 답을 맞혀왔는데, 정작 실전에는 읽을 미래가 없는 것입니다.

> **loss 하나만 보고 모델이 좋다고 판단하면 안 되는 이유**를 이보다 잘 보여주는 예는 드뭅니다.
> 반드시 **실제로 하려던 일**(여기서는 문장 생성)을 시켜봐야 합니다.

---

## 연습 문제 3 — 위치 임베딩을 빼보기

> `+ self.pos_emb(pos)`를 지우고 학습시켜보세요.
> 6절에서 "순서를 모른다"고 했으니 크게 망가질 것 같은데, 실제로 해보면 예상과 다릅니다.

In [ ]:
no_pos_model, val_nopos = train("위치 임베딩 없음", use_pos=False)

print()
print(f"기준            : {base_val:.4f}")
print(f"위치 임베딩 없음: {val_nopos:.4f}   (차이 {val_nopos - base_val:+.4f})")
print()
print(generate(no_pos_model, "월요일에 ", 200))

### 해설

제가 돌렸을 때 **0.1923**입니다. 기준(0.1922)과 **소수점 넷째 자리까지 같습니다.**
6절에서 "어텐션은 순서를 모른다"고 실험까지 해놓고 이게 무슨 일일까요?

**힌트는 4절의 인과 마스크였습니다.**

6절의 실험에서는 **마스크를 껐습니다**("순서 효과만 보려고 마스크는 잠시 끕니다"라고 적혀 있습니다).
마스크가 없을 때만 어텐션이 순서에 완전히 무관합니다. **마스크를 켜면 이야기가 달라집니다.**

```text
0번 자리: 볼 수 있는 글자 1개
1번 자리: 볼 수 있는 글자 2개
2번 자리: 볼 수 있는 글자 3개
...
```

**자리마다 볼 수 있는 글자 수가 다릅니다.** 즉 마스크 자체가 "너는 몇 번째다"라는 정보를
간접적으로 흘리고 있고, 모델은 그것을 위치 단서로 쓸 수 있습니다.
[실제로 알려진 현상](https://arxiv.org/abs/2203.16634)입니다.

### 그런데 생성 결과를 보면 얘기가 다릅니다

**loss는 같은데 생성 문장에는 실수가 섞입니다.** 제가 돌렸을 때 이런 줄이 나왔습니다.

```text
화요일에 서연은 김밥을 피자를 먹었습니다. 먹은 것은 김치입니다.
수요일에 유진이 먹었습니다. 하나가
```

음식이 두 번 나오고, 이름이 통째로 빠지고, 앞뒤 음식이 안 맞습니다.
기준 모델의 생성에서는 이런 것이 거의 안 나옵니다.

**왜 loss에는 안 잡혔을까요?** loss는 **모든 위치의 평균**이기 때문입니다.
문장 대부분의 자리는 위치 정보 없이도 잘 맞히고, 틀리는 것은 몇 자리뿐입니다.
평균을 내면 그 차이가 소수점 아래로 묻힙니다.

**연습 문제 2번과 정확히 같은 교훈입니다.** 거기서는 loss가 좋은데 생성이 엉망이었고,
여기서는 loss가 같은데 생성이 조금 나쁩니다. **loss는 요약이지 전부가 아닙니다.**

**그리고 이 결과를 일반화하지는 마세요.** 우리 데이터는 모든 문장이 같은 틀을 가진
**아주 규칙적인** 데이터라 특히 쉬운 경우입니다. 진짜 언어에서는 위치 표현을 잘 만드는 것이
성능에 크게 영향을 줍니다. 요즘 모델이 [RoPE](https://github.com/karzit/temp/blob/master/glossary.md#rope) 같은 방식을 따로 개발해 쓰는 이유입니다.

> **여기서 배울 점**: "A를 빼도 loss가 그대로다 → A는 필요 없다"가 **아닙니다.**
> ① 다른 곳(마스크)에서 같은 정보가 새어 들어오고 있는지,
> ② loss 말고 **실제로 하려던 일**에서도 정말 같은지 — 둘 다 확인해야 합니다.

---

## 연습 문제 4 — 온도 바꿔보기

> `temperature`를 0.1, 0.8, 1.5로 바꿔 생성 결과를 비교하세요.
> 어느 쪽이 문법을 더 잘 지키나요? 그리고 그것이 항상 좋은 걸까요?

In [ ]:
for t in [0.1, 0.8, 1.5]:
    print(f"===== temperature = {t} =====")
    print(generate(base_model, "월요일에 ", 150, temperature=t))
    print()

### 해설

**`0.1` — 거의 안 틀립니다. 그리고 거의 안 변합니다.**
점수를 0.1로 나누면 1등과 2등의 차이가 10배로 벌어져, 사실상 **항상 1등만 고릅니다.**
문법은 완벽하지만 같은 요일·같은 이름·같은 음식이 계속 반복됩니다.

**`0.8` — 문법을 지키면서 내용은 다양합니다.** 본문에서 쓴 기본값입니다.

**`1.5` — 다양하지만 틀리기 시작합니다.** 점수 차이가 눌려서 확률이 평평해집니다.
조사가 틀리거나(`김밥를`), 앞뒤 음식이 안 맞거나, 문장이 깨진 것이 섞입니다.

### "그것이 항상 좋은 걸까?"

**아닙니다. 무엇을 하려는지에 따라 다릅니다.**

| 하려는 일 | 적당한 온도 | 이유 |
|---|---|---|
| 문서에서 사실 뽑아내기, 분류, 정형 출력 | **낮게** (0~0.3) | 매번 같은 답이 나와야 한다. 창의성은 방해가 된다 |
| 요약, 번역 | 중간 (0.3~0.7) | 정확해야 하지만 문장은 자연스러워야 한다 |
| 아이디어 내기, 카피 쓰기 | 높게 (0.8~1.2) | 뻔한 1등 말고 다른 것이 필요하다 |

`rag-pipeline-practice` 03번에서 [정형 출력](https://github.com/karzit/temp/blob/master/glossary.md#structured-output)을 만들 때 온도를 낮게 잡는 것이
정확히 이 이유입니다. 같은 문서를 두 번 넣었는데 다른 JSON이 나오면 곤란하니까요.

**그리고 온도가 0이어도 틀린 답이 안 나오는 것은 아닙니다.** 온도는 "얼마나 다양하게 뽑을지"만
정합니다. **모델이 애초에 잘못 알고 있으면, 그 잘못된 답을 아주 일관되게 내놓을 뿐입니다.**

---

## 정리

네 문제에서 얻은 것을 한 줄씩 적으면 이렇습니다.

1. **한 번 돌린 결과로 조건을 비교하면 안 된다** — 조건 간 차이가 실행 간 흔들림보다 작으면
   아무 말도 할 수 없다. 그리고 평균만큼이나 **흔들림**이 중요할 때가 있다(1헤드).
2. **loss가 하한보다 낮으면 부정행위다** — 하한을 미리 계산해두면 잡아낼 수 있다.
   그리고 loss가 좋아도 **실제로 시켜봐야** 안다.
3. **loss가 같다고 같은 모델이 아니다** — 위치 임베딩을 빼도 loss는 그대로였지만
   생성 문장에는 실수가 늘었다. loss는 모든 위치의 평균이라 몇 자리의 실수를 묻어버린다.
4. **온도에 정답은 없다** — 하려는 일에 맞춰 고르는 값이다.

1~3번은 전부 같은 이야기의 다른 얼굴입니다 — **숫자 하나를 보고 결론 내지 않기.**
이 저장소가 여러 노트북에서 반복하는 것(기준선 만들기, 교차 검증, 데이터 누출 확인,
혼동 행렬 보기)과 같은 습관입니다.

**돌아가기**: [07_transformer.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/07_transformer/07_transformer.ipynb)